In [ ]:
import json

import gc_utils
import gizmo_analysis as gizmo
import matplotlib.pyplot as plt
import numpy as np
import numpy.ma as ma
import pandas as pd
from matplotlib import colors
from tqdm import tqdm

In [ ]:
sim = "m12c"
# sim_dir = "/Users/z5114326/Documents/simulations/"
sim_dir = "/Volumes/Expansion/simulations/"

sim_codes = sim_dir + "simulation_codes.json"
with open(sim_codes) as sim_json:
    sim_data = json.load(sim_json)


pub_data = sim_dir + "snapshot_times_public.txt"
pub_snaps = pd.read_table(pub_data, comment="#", header=None, sep=r"\s+")
pub_snaps.columns = [
    "index",
    "scale_factor",
    "redshift",
    "time_Gyr",
    "lookback_time_Gyr",
    "time_width_Myr",
]

In [ ]:
fire_dir = sim_dir + sim + "/" + sim + "_res7100/"
# halt = gc_utils.get_halo_tree(sim, sim_dir, species="star")

Retrieving Snapshot 600..................: 100%|████████████████████████████████████████████████████████████████████████| 1/1 [00:21<00:00, 21.72s/it]


In [ ]:
# --- Snapshot range ---
snap_min = 214
snap_max = 446
snap_lst = pub_snaps[(snap_min <= pub_snaps["index"]) & (pub_snaps["index"] <= snap_max)]["index"].values

# --- Spatial binning parameters ---
r_lim = 10  # kpc
distance_bin_width = 0.1  # kpc
bin_edges = np.arange(-r_lim, r_lim + distance_bin_width, distance_bin_width)
n_bins = len(bin_edges) - 1

# --- Arrays to accumulate SFR and counts ---
sfr_sum = np.zeros((n_bins, n_bins))
count_sum = np.zeros((n_bins, n_bins))

# --- Loop over snapshots ---
for snap in tqdm(snap_lst, desc="Processing snapshots"):
    part = gc_utils.open_snapshot(snap, fire_dir, species=["gas"])

    # Positions
    pos_cart = part["gas"].prop("host.distance.principal")
    pos_cyl = part["gas"].prop("host.distance.principal.cylindrical")

    # Select gas near the midplane and within r_lim
    r_msk = pos_cyl[:, 0] <= r_lim
    z_msk = np.abs(pos_cyl[:, 2]) <= 1.0  # ±1 kpc
    msk = r_msk & z_msk

    pos = pos_cart[msk]
    sfr = part["gas"]["sfr"][msk]

    # 2D histogram: total SFR per (x, y) bin
    sfr_hist, _, _ = np.histogram2d(pos[:, 0], pos[:, 1], bins=[bin_edges, bin_edges], weights=sfr)

    # 2D histogram: particle count per (x, y) bin
    count_hist, _, _ = np.histogram2d(pos[:, 0], pos[:, 1], bins=[bin_edges, bin_edges])

    # Accumulate sums across all snapshots
    sfr_sum += sfr_hist
    count_sum += count_hist

# --- Compute spatially averaged SFR ---
sfr_mean = np.divide(sfr_sum, count_sum, out=np.zeros_like(sfr_sum), where=count_sum > 0)

# --- Plot ---
plt.figure(figsize=(8, 7))
extent = [-r_lim, r_lim, -r_lim, r_lim]
plt.imshow(np.log10(sfr_mean + 1e-12).T, origin="lower", extent=extent, aspect="equal", cmap="magma")
plt.colorbar(label="log10(<SFR>)")
plt.xlabel("x [kpc]")
plt.ylabel("y [kpc]")
plt.title(f"Spatially averaged SFR across snapshots {snap_min}–{snap_max}")